In [56]:
import pandas as pd , numpy as np 
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit , cross_val_score 
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

data=pd.read_csv("practicing.csv")
data.head()
data_2=data.copy()
data_2

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25,1665,374.0,845,330,1.5603,78100,INLAND
20636,-121.21,39.49,18,697,150.0,356,114,2.5568,77100,INLAND
20637,-121.22,39.43,17,2254,485.0,1007,433,1.7000,92300,INLAND
20638,-121.32,39.43,18,1860,409.0,741,349,1.8672,84700,INLAND


In [20]:
# We can fill the gap via pd or directly use the feature of machine learning via creating a pipeline
data_copy=data.copy()
data_copy['total_bedrooms']=data_copy['total_bedrooms'].fillna(data['total_bedrooms'].mean())
data_copy.isnull().sum()
data_copy['total_bedrooms'].isnull().sum()


np.int64(0)

In [57]:
data_2_features=data_2.drop("ocean_proximity",axis=1)
data_2_labels=data_2[['ocean_proximity']]

number_pipeline=Pipeline([
    ['Impute',SimpleImputer(strategy='mean')]
])

features_pipeline=Pipeline([
    ['Encoder',OneHotEncoder(handle_unknown='ignore')]
    
])
full_pipeline=ColumnTransformer([
    ('numbers',number_pipeline,data_2_features.columns),
    ('features',features_pipeline,['ocean_proximity'])
])
converting=full_pipeline.fit_transform(data_2)
dataframing=pd.DataFrame(converting,columns=full_pipeline.get_feature_names_out(),index=data_2.index)
dataframing
data_2_labels

# Do concatination for the splitting the size
concatination=pd.concat([dataframing,data_2_labels],axis=1)
concatination

# Now the data visualization is present here and we will do 
# import matplotlib.pyplot as plt 
# dataframing.plot(kind='scatter',x='numbers__latitude',y='numbers__longitude',cmap='inferno',c='numbers__latitude')
# plt.grid(True)
# plt.title("Housing")
# plt.show()

,numbers__longitude,numbers__latitude,numbers__housing_median_age,numbers__total_rooms,numbers__total_bedrooms,numbers__population,numbers__households,numbers__median_income,numbers__median_house_value,features__ocean_proximity_<1H OCEAN,features__ocean_proximity_INLAND,features__ocean_proximity_ISLAND,features__ocean_proximity_NEAR BAY,features__ocean_proximity_NEAR OCEAN,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,0.0,0.0,0.0,1.0,0.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,0.0,0.0,0.0,1.0,0.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,0.0,0.0,0.0,1.0,0.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,0.0,0.0,0.0,1.0,0.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,0.0,0.0,0.0,1.0,0.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,0.0,1.0,0.0,0.0,0.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,0.0,1.0,0.0,0.0,0.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,0.0,1.0,0.0,0.0,0.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,0.0,1.0,0.0,0.0,0.0,INLAND


In [85]:
# Now split the data on which we have done all the work for better kinda work
splitting=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
#I forget here for pd.cut so all the numbers come under both test and train and the string cannot come via that remember
concatination['number_string']=pd.cut(concatination['numbers__median_income'],bins=[-1,1.5,3,4.5,6,7.5,np.inf],labels=[1,2,3,4,5,6])
for x , y in splitting.split(concatination,concatination['number_string']):
    # We do like this because we want to match the numbers 1,2,3,4,5,6 in both and string variable does not affect it at all
    test_size=concatination.iloc[x]
    train_size=concatination.iloc[y]

train_size.reset_index(drop=True,inplace=True)
train_size.isnull().sum()
train_size


# After doing all , means inserting in the pipelines of different and doing split , we go and check for the models answers
# and how wrong the answers can be and we have 3 models for it 
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
# Now comes the checking / evaluation of the model and at last , do work on GIT and push
model=LinearRegression()
labels=train_size[['ocean_proximity']]
remove_train_size=train_size.drop(['ocean_proximity'],axis=1)
first=remove_train_size.drop(['numbers__median_house_value'],axis=1)
second=remove_train_size[['numbers__median_house_value']]
modelling=model.fit(first,second)



C:\Users\Jenil\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


ValueError: Expected 2D array, got scalar array instead:
array=LinearRegression().
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.